# CUMULUS: Stage-Level Evaluation of Eye-Tracking Preprocessing

This notebook reproduces the **stage-level evaluation** described in *Robust Preprocessing Pipelines for Eye-Tracking Data* by Jennifer Landes and Meike Klettke.

The notebook follows the methodological order of the paper and evaluates the three preprocessing stages separately:

1. **Missing-value imputation**: Mean, LOCF, KNN (`k = 5`)
2. **Outlier handling**: Z-score (`|z| > 3`), MAD, Isolation Forest
3. **Normalization**: Min-Max, Z-score, Robust scaling

Controlled missingness and spike corruptions are injected at **5%, 10%, 15%, and 20%** using fixed random seeds. Metrics are computed **per file and per feature** and are then aggregated to dataset-level summaries.

> **Scope.** This notebook intentionally stops at the individual preprocessing stages. The evaluation of complete preprocessing pipelines is kept separate and belongs in the second reproduction notebook.


## 1. Reproducibility and configuration

The default paths below correspond to the original local experiment setup. They can be overridden without editing the notebook by setting the environment variables `CUMULUS_D2_DIR` and `CUMULUS_CHEATING_DIR`.

The benchmark writes one Excel workbook per dataset. Each workbook contains the run log, raw per-file metric rows, and the corresponding aggregated summary tables.


In [ ]:
from __future__ import annotations

import hashlib
import os
import platform
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import sklearn
from scipy.stats import ks_2samp, kurtosis, median_abs_deviation, skew, wilcoxon, zscore
from sklearn.ensemble import IsolationForest
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import MinMaxScaler, RobustScaler, StandardScaler

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

# Original local dataset locations; environment variables can override them.
D2_DIR = Path(os.environ.get(
    "CUMULUS_D2_DIR",
    r"C:\Users\oxije\Dropbox\Dissertation\Daten\01_Datensaetze_Original\02_Autism_EyeTracking\Eye-tracking Output",
))
CHEATING_DIR = Path(os.environ.get(
    "CUMULUS_CHEATING_DIR",
    r"C:\Users\oxije\Dropbox\Dissertation\Daten\02_Eigenes_Experiment\Cheating_Detection_Experiment\Originaldaten_split\alle\eyetracking_Cheating",
))

PROJECT_ROOT = Path.cwd()
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURE_DIR = PROJECT_ROOT / "figures"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

BASE_SEED = 42
MISSING_LEVELS = (5, 10, 15, 20)
OUTLIER_LEVELS = (5, 10, 15, 20)

ZSCORE_THRESHOLD = 3.0
MAD_THRESHOLD = 3.0
IFOREST_CONTAMINATION = 0.05

FEATURE_COLS = ["Gaze X", "Gaze Y", "ET_PupilLeft", "ET_PupilRight"]
IMPUTATION_METHODS = ("mean", "locf", "knn(k=5)")
OUTLIER_METHODS = ("zscore", "mad", "iforest")
SCALERS = ("minmax", "zscore", "robust")

# The stage-level normalization output used in the original benchmark retains the
# two temporal/multivariate imputation candidates and the two robust outlier candidates.
NORMALIZATION_IMPUTERS = ("knn(k=5)", "locf")
NORMALIZATION_OUTLIERS = ("iforest", "mad")

print(f"Python:       {sys.version.split()[0]}")
print(f"Platform:     {platform.platform()}")
print(f"NumPy:        {np.__version__}")
print(f"pandas:       {pd.__version__}")
print(f"SciPy:        {scipy.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"D2 data:      {D2_DIR}")
print(f"Cheating:     {CHEATING_DIR}")
print(f"Results:      {RESULTS_DIR.resolve()}")


## 2. Datasets and canonical signal representation

CUMULUS evaluates two heterogeneous eye-tracking datasets: the in-house **Cheating dataset** and the public **D2 Autism Eye-Tracking dataset**.

To make the benchmark independent of vendor-specific exports, every recording is mapped to the canonical four-dimensional signal representation

\[
X_t = (\mathrm{GazeX}_t,\; \mathrm{GazeY}_t,\; \mathrm{PupilL}_t,\; \mathrm{PupilR}_t).
\]

Invalid measurements and tracking losses are represented as `NaN`. If only monocular pupil information is available, it is mapped to the left-pupil channel and the unavailable right-pupil channel remains missing. Binocular gaze coordinates are averaged per time step when necessary.


In [ ]:
def find_data_files(root: Path) -> list[Path]:
    """Return CSV/TSV recordings in deterministic filename order."""
    files = list(root.glob("*.csv")) + list(root.glob("*.tsv"))
    return sorted(p for p in files if p.is_file())


def _read_table(path: Path) -> pd.DataFrame:
    """Read a delimited eye-tracking export while tolerating common separators."""
    if path.suffix.lower() == ".tsv":
        return pd.read_csv(path, sep="\t", low_memory=False)

    # Most source files are comma-separated. A fallback is useful for exported
    # semicolon-separated CSV files without changing the benchmark logic.
    df = pd.read_csv(path, low_memory=False)
    if len(df.columns) == 1:
        df = pd.read_csv(path, sep=";", low_memory=False)
    return df


def to_numeric_signal(series: pd.Series) -> pd.Series:
    """Convert vendor-formatted numeric columns to floating-point signals."""
    return pd.to_numeric(
        series.astype(str)
        .str.replace(",", ".", regex=False)
        .replace({"-": np.nan, "nan": np.nan, "None": np.nan, "": np.nan}),
        errors="coerce",
    )


def load_eye_file(path: Path) -> pd.DataFrame:
    """Map supported eye-tracking exports to the canonical CUMULUS representation."""
    df = _read_table(path)
    cols = set(df.columns)

    # A) Already canonical / Cheating export.
    if set(FEATURE_COLS).issubset(cols):
        out = df[FEATURE_COLS].copy()
        for col in FEATURE_COLS:
            out[col] = to_numeric_signal(out[col])
        return out.reset_index(drop=True)

    # B) Tobii-style export.
    tobii = {"Gaze point X", "Gaze point Y", "Pupil diameter left", "Pupil diameter right"}
    if tobii.issubset(cols):
        work = df.copy()
        if "Sensor" in work.columns:
            work = work[work["Sensor"].astype(str).eq("Eye Tracker")].copy()
        if "Event" in work.columns:
            event = work["Event"]
            work = work[event.isna() | event.astype(str).eq("")].copy()

        out = work[["Gaze point X", "Gaze point Y", "Pupil diameter left", "Pupil diameter right"]].copy()
        for col in out.columns:
            out[col] = to_numeric_signal(out[col])

        if "Validity left" in work.columns and "Validity right" in work.columns:
            valid_l = work["Validity left"].astype(str)
            valid_r = work["Validity right"].astype(str)
            out.loc[valid_l.eq("Invalid"), "Pupil diameter left"] = np.nan
            out.loc[valid_r.eq("Invalid"), "Pupil diameter right"] = np.nan
            both_invalid = valid_l.eq("Invalid") & valid_r.eq("Invalid")
            out.loc[both_invalid, ["Gaze point X", "Gaze point Y"]] = np.nan

        out = out.rename(columns={
            "Gaze point X": "Gaze X",
            "Gaze point Y": "Gaze Y",
            "Pupil diameter left": "ET_PupilLeft",
            "Pupil diameter right": "ET_PupilRight",
        })
        return out[FEATURE_COLS].reset_index(drop=True)

    # C) SMI / point-of-regard export.
    por = {
        "rx": "Point of Regard Right X [px]",
        "ry": "Point of Regard Right Y [px]",
        "lx": "Point of Regard Left X [px]",
        "ly": "Point of Regard Left Y [px]",
        "pr": "Pupil Diameter Right [mm]",
        "pl": "Pupil Diameter Left [mm]",
    }
    if set(por.values()).issubset(cols):
        gx_r = to_numeric_signal(df[por["rx"]])
        gy_r = to_numeric_signal(df[por["ry"]])
        gx_l = to_numeric_signal(df[por["lx"]])
        gy_l = to_numeric_signal(df[por["ly"]])
        return pd.DataFrame({
            "Gaze X": pd.concat([gx_l, gx_r], axis=1).mean(axis=1, skipna=True),
            "Gaze Y": pd.concat([gy_l, gy_r], axis=1).mean(axis=1, skipna=True),
            "ET_PupilLeft": to_numeric_signal(df[por["pl"]]),
            "ET_PupilRight": to_numeric_signal(df[por["pr"]]),
        })[FEATURE_COLS].reset_index(drop=True)

    # D) Generic binocular raw export.
    binocular = {"gaze_x_left", "gaze_y_left", "gaze_x_right", "gaze_y_right", "pupil_left", "pupil_right"}
    if binocular.issubset(cols):
        gx_l = to_numeric_signal(df["gaze_x_left"])
        gy_l = to_numeric_signal(df["gaze_y_left"])
        gx_r = to_numeric_signal(df["gaze_x_right"])
        gy_r = to_numeric_signal(df["gaze_y_right"])
        return pd.DataFrame({
            "Gaze X": pd.concat([gx_l, gx_r], axis=1).mean(axis=1, skipna=True),
            "Gaze Y": pd.concat([gy_l, gy_r], axis=1).mean(axis=1, skipna=True),
            "ET_PupilLeft": to_numeric_signal(df["pupil_left"]),
            "ET_PupilRight": to_numeric_signal(df["pupil_right"]),
        })[FEATURE_COLS].reset_index(drop=True)

    # E) Generic monocular export.
    if {"x", "y", "pupil"}.issubset(cols):
        out = pd.DataFrame({
            "Gaze X": to_numeric_signal(df["x"]),
            "Gaze Y": to_numeric_signal(df["y"]),
            "ET_PupilLeft": to_numeric_signal(df["pupil"]),
            "ET_PupilRight": np.nan,
        })
        if "missing" in df.columns:
            missing = pd.to_numeric(df["missing"], errors="coerce").fillna(0).astype(int).eq(1)
            out.loc[missing, ["Gaze X", "Gaze Y", "ET_PupilLeft"]] = np.nan
        return out[FEATURE_COLS].reset_index(drop=True)

    raise KeyError(
        f"Unsupported eye-tracking format in {path.name}. "
        f"First columns: {list(df.columns)[:40]}"
    )


def dataset_inventory(name: str, root: Path) -> pd.DataFrame:
    files = find_data_files(root)
    rows = []
    for path in files:
        try:
            signal = load_eye_file(path)
            rows.append({
                "dataset": name,
                "file": path.name,
                "rows": len(signal),
                "status": "ok",
                **{f"missing_{c}": float(signal[c].isna().mean()) for c in FEATURE_COLS},
            })
        except Exception as exc:
            rows.append({"dataset": name, "file": path.name, "rows": np.nan, "status": repr(exc)})
    return pd.DataFrame(rows)


In [ ]:
# Optional preflight: this checks only that the paths and source formats are readable.
for dataset_name, dataset_dir in [("D2", D2_DIR), ("Cheating", CHEATING_DIR)]:
    files = find_data_files(dataset_dir)
    print(f"{dataset_name}: {len(files)} source files")
    if not files:
        print(f"  WARNING: no CSV/TSV files found under {dataset_dir}")
    else:
        sample = load_eye_file(files[0])
        print(f"  sample={files[0].name!r}, rows={len(sample):,}, missing={sample.isna().mean().round(3).to_dict()}")


## 3. Preprocessing design space

The paper defines three sequential preprocessing stages:

| Stage | Evaluated methods |
|---|---|
| Missing-value imputation | Mean; LOCF; KNN (`k = 5`) |
| Outlier handling | Z-score filtering (`|z| > 3`); MAD filtering; Isolation Forest |
| Normalization | Min-Max `[0,1]`; Z-score standardization; Robust scaling (median/IQR) |

The complete-pipeline interaction analysis is deliberately deferred to the second notebook. The present notebook evaluates the individual stages and produces the stage-level result workbooks.

![Figure 1 - Combination space of preprocessing pipelines](figures/figure_1_pipeline_design_space.png)

*Figure 1. Combination space of preprocessing pipelines across the preprocessing stages (original figure from the paper).*


## 4. Controlled corruption models

Two reproducible corruption models are used:

### MCAR missingness injection
For each signal feature independently, a proportion

\[
p \in \{5,10,15,20\}\% 
\]

of sample indices is selected and replaced by `NaN`. Because the uncorrupted values are retained as a reference, the reconstruction error can be measured exactly at the synthetically masked positions.

### Spike outlier injection
For each selected sample, a positive spike proportional to the feature standard deviation is added:

\[
x_t \leftarrow x_t + \lambda \sigma_x, \qquad \lambda \sim U(5,10).
\]

A fixed seed is re-used across candidate methods, ensuring that every method is evaluated on the same corruption pattern.


In [ ]:
def inject_missing_mcar(df: pd.DataFrame, pct: float, seed: int = BASE_SEED) -> pd.DataFrame:
    """Inject feature-wise MCAR missingness using the benchmark's fixed-seed protocol."""
    rng = np.random.default_rng(seed)
    out = df.copy()
    n = len(out)
    for col in FEATURE_COLS:
        k = int(n * pct / 100.0)
        if k <= 0:
            continue
        idx = rng.choice(out.index.to_numpy(), size=k, replace=False)
        out.loc[idx, col] = np.nan
    return out


def inject_spikes(df: pd.DataFrame, pct: float, seed: int = BASE_SEED) -> pd.DataFrame:
    """Inject spike artifacts x_t <- x_t + lambda*sigma, lambda ~ Uniform(5,10)."""
    rng = np.random.default_rng(seed)
    out = df.copy()
    n = len(out)
    for col in FEATURE_COLS:
        k = int(n * pct / 100.0)
        if k <= 0:
            continue
        idx = rng.choice(out.index.to_numpy(), size=k, replace=False)
        sigma = np.nanstd(out[col].to_numpy(dtype=float))
        if not np.isfinite(sigma) or sigma == 0:
            continue
        lam = rng.uniform(5.0, 10.0, size=k)
        out.loc[idx, col] = out.loc[idx, col].to_numpy(dtype=float) + lam * sigma
    return out


## 5. Preprocessing methods

The implementations below are intentionally compact and deterministic. Imputation is applied feature-wise/multivariately as appropriate; outlier methods replace detected samples by `NaN`; and normalization is applied feature-wise while preserving missing positions.

For LOCF, forward filling implements the temporal propagation step. A backward fill is used only to resolve leading boundary gaps so that synthetically removed values at the beginning of a recording remain scorable.


In [ ]:
def mean_impute(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    active = [c for c in FEATURE_COLS if out[c].notna().any()]
    if active:
        out[active] = SimpleImputer(strategy="mean").fit_transform(out[active])
    return out


def locf_impute(df: pd.DataFrame) -> pd.DataFrame:
    return df.ffill().bfill()


def knn_impute(df: pd.DataFrame, k: int = 5) -> pd.DataFrame:
    out = df.copy()
    active = [c for c in FEATURE_COLS if out[c].notna().any()]
    if active:
        out[active] = KNNImputer(n_neighbors=k).fit_transform(out[active])
    return out


def apply_imputer(df: pd.DataFrame, method: str) -> pd.DataFrame:
    if method == "mean":
        return mean_impute(df)
    if method == "locf":
        return locf_impute(df)
    if method == "knn(k=5)":
        return knn_impute(df, k=5)
    raise ValueError(f"Unknown imputation method: {method}")


def outliers_to_nan(df: pd.DataFrame, method: str) -> pd.DataFrame:
    out = df.copy()

    for col in FEATURE_COLS:
        x = out[col].to_numpy(dtype=float)
        valid = np.isfinite(x)
        if valid.sum() < 5:
            continue

        if method == "zscore":
            scores = np.full(len(x), np.nan)
            scores[valid] = np.abs(zscore(x[valid], nan_policy="omit"))
            mask = scores > ZSCORE_THRESHOLD

        elif method == "mad":
            median = np.nanmedian(x)
            mad = median_abs_deviation(x[valid], nan_policy="omit")
            if not np.isfinite(mad) or mad == 0:
                mask = np.zeros(len(x), dtype=bool)
            else:
                mask = np.abs(x - median) > MAD_THRESHOLD * mad

        elif method == "iforest":
            mask = np.zeros(len(x), dtype=bool)
            detector = IsolationForest(
                contamination=IFOREST_CONTAMINATION,
                random_state=BASE_SEED,
                n_jobs=-1,
            )
            pred = detector.fit_predict(x[valid].reshape(-1, 1))
            mask[np.where(valid)[0][pred == -1]] = True

        else:
            raise ValueError(f"Unknown outlier method: {method}")

        out.loc[out.index[mask], col] = np.nan

    return out


def scale_series(series: pd.Series, method: str) -> pd.Series:
    """Scale one feature while retaining NaN positions."""
    x = pd.to_numeric(series, errors="coerce").astype(float)
    valid = x.notna()
    result = pd.Series(np.nan, index=x.index, dtype=float)
    if valid.sum() == 0:
        return result

    if method == "minmax":
        scaler = MinMaxScaler(feature_range=(0, 1))
    elif method == "zscore":
        scaler = StandardScaler()
    elif method == "robust":
        scaler = RobustScaler()
    else:
        raise ValueError(f"Unknown scaler: {method}")

    result.loc[valid] = scaler.fit_transform(x.loc[valid].to_numpy().reshape(-1, 1)).ravel()
    return result


def apply_scaler(df: pd.DataFrame, method: str) -> pd.DataFrame:
    return pd.DataFrame({col: scale_series(df[col], method) for col in FEATURE_COLS}, index=df.index)


## 6. Evaluation protocol and metrics

The benchmark separates **reconstruction accuracy** from **distributional stability**.

- **Imputation:** RMSE and MAE are calculated only at synthetically masked positions. A Wilcoxon signed-rank test is retained as an additional paired comparison.
- **Outlier handling:** the cleaned signal is compared with the uncorrupted reference using the Kolmogorov-Smirnov statistic and variance reduction.
- **Normalization:** distribution preservation is summarized using KS distance and changes in variance, skewness, and kurtosis. Reference and processed signals are placed on the same normalized coordinate system before the comparison; this avoids comparing raw pixel/mm units directly with `[0,1]` or standardized values.

All metrics are computed per recording and per feature before dataset-level aggregation.

![Figure 2 - CUMULUS evaluation workflow](figures/figure_2_cumulus_workflow.png)

*Figure 2. CUMULUS workflow for evaluating preprocessing pipelines under controlled corruption (original figure from the paper).*


In [ ]:
def safe_wilcoxon(a, b) -> tuple[float, float]:
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    ok = np.isfinite(a) & np.isfinite(b)
    a, b = a[ok], b[ok]
    if len(a) < 5:
        return np.nan, np.nan
    if np.allclose(a - b, 0):
        return 0.0, 1.0
    try:
        stat, p = wilcoxon(a, b)
        return float(stat), float(p)
    except Exception:
        return np.nan, np.nan


def reconstruction_metrics(reference: pd.DataFrame, corrupted: pd.DataFrame, reconstructed: pd.DataFrame, feature: str) -> dict:
    mask = reference[feature].notna() & corrupted[feature].isna()
    y_true = reference.loc[mask, feature].to_numpy(dtype=float)
    y_pred = reconstructed.loc[mask, feature].to_numpy(dtype=float)
    ok = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true, y_pred = y_true[ok], y_pred[ok]

    if len(y_true) == 0:
        return {"rmse": np.nan, "mae": np.nan, "wilcoxon_stat": np.nan, "wilcoxon_p": np.nan}

    stat, p = safe_wilcoxon(y_true, y_pred)
    return {
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "wilcoxon_stat": stat,
        "wilcoxon_p": p,
    }


def distribution_metrics(reference: pd.Series, processed: pd.Series) -> dict:
    a = pd.to_numeric(reference, errors="coerce").dropna().astype(float)
    b = pd.to_numeric(processed, errors="coerce").dropna().astype(float)
    if len(a) < 5 or len(b) < 5:
        return {
            "variance_reduction_pct": np.nan,
            "skewness_reduction": np.nan,
            "kurtosis_reduction": np.nan,
            "ks_stat": np.nan,
            "ks_p": np.nan,
        }

    var_a, var_b = float(np.var(a)), float(np.var(b))
    skew_a, skew_b = float(skew(a)), float(skew(b))
    kurt_a, kurt_b = float(kurtosis(a)), float(kurtosis(b))
    ks = ks_2samp(a, b)

    return {
        "variance_reduction_pct": np.nan if var_a == 0 else (var_a - var_b) / var_a * 100.0,
        "skewness_reduction": abs(skew_b - skew_a),
        "kurtosis_reduction": abs(kurt_b - kurt_a),
        "ks_stat": float(ks.statistic),
        "ks_p": float(ks.pvalue),
    }


def normalization_metrics(reference: pd.Series, processed: pd.Series, scaler: str) -> dict:
    """Compare reference and processed distributions after applying the same scaler class."""
    ref_scaled = scale_series(reference, scaler)
    proc_scaled = scale_series(processed, scaler)
    return distribution_metrics(ref_scaled, proc_scaled)


## 7. Stage 1 - Missing-value imputation

For each missingness level, the same corrupted recording is passed to all three candidate methods. RMSE and MAE are evaluated only at locations that were observed in the reference and became missing after controlled masking.


In [ ]:
def evaluate_imputation_file(reference: pd.DataFrame, file_name: str) -> pd.DataFrame:
    rows = []
    for level in MISSING_LEVELS:
        corrupted = inject_missing_mcar(reference, level, seed=BASE_SEED)
        for method in IMPUTATION_METHODS:
            reconstructed = apply_imputer(corrupted, method)
            for feature in FEATURE_COLS:
                metrics = reconstruction_metrics(reference, corrupted, reconstructed, feature)
                rows.append({
                    "file": file_name,
                    "missing_level": level,
                    "method": method,
                    "feature": feature,
                    **metrics,
                })
    return pd.DataFrame(rows)


## 8. Stage 2 - Outlier handling

Spike artifacts are injected independently of the missing-value experiment. Each candidate outlier method is applied to the same spike-corrupted signal, and the cleaned result is compared with the uncorrupted reference distribution.


In [ ]:
def evaluate_outlier_file(reference: pd.DataFrame, file_name: str) -> pd.DataFrame:
    rows = []
    for level in OUTLIER_LEVELS:
        corrupted = inject_spikes(reference, level, seed=BASE_SEED)
        for method in OUTLIER_METHODS:
            cleaned = outliers_to_nan(corrupted, method)
            for feature in FEATURE_COLS:
                metrics = distribution_metrics(reference[feature], cleaned[feature])
                rows.append({
                    "file": file_name,
                    "outlier_level": level,
                    "method": method,
                    "feature": feature,
                    **metrics,
                })
    return pd.DataFrame(rows)


## 9. Stage 3 - Normalization

Normalization is evaluated after applying the retained upstream preprocessing candidates to the original recording. This stage does **not** introduce an additional synthetic corruption. Each normalized processed signal is compared with the corresponding normalized reference signal using distributional metrics.


In [ ]:
def evaluate_normalization_file(reference: pd.DataFrame, file_name: str) -> pd.DataFrame:
    rows = []

    for mv_method in NORMALIZATION_IMPUTERS:
        imputed = apply_imputer(reference, mv_method)

        for outlier_method in NORMALIZATION_OUTLIERS:
            cleaned = outliers_to_nan(imputed, outlier_method)

            for scaler in SCALERS:
                for feature in FEATURE_COLS:
                    metrics = normalization_metrics(reference[feature], cleaned[feature], scaler)
                    rows.append({
                        "file": file_name,
                        "mv_method": mv_method,
                        "outlier_method": outlier_method,
                        "scaler": scaler,
                        "feature": feature,
                        **metrics,
                    })

    return pd.DataFrame(rows)


## 10. Dataset-level execution and aggregation

Each source recording is processed independently. A failed file is recorded in the run log and does not prevent the remaining recordings from being evaluated.

The summary tables are arithmetic means over the available per-file metric rows for each stage configuration, matching the paper's per-file/per-feature aggregation protocol.


In [ ]:
def run_stage_benchmark(dataset_name: str, root: Path) -> dict[str, pd.DataFrame]:
    files = find_data_files(root)
    if not files:
        raise FileNotFoundError(f"No CSV/TSV files found for {dataset_name}: {root}")

    logs = []
    imp_parts = []
    out_parts = []
    norm_parts = []

    print(f"\n=== {dataset_name}: {len(files)} files ===")

    for idx, path in enumerate(files, start=1):
        print(f"[{idx:>3}/{len(files)}] {path.name}")
        try:
            reference = load_eye_file(path).apply(pd.to_numeric, errors="coerce")

            imp_parts.append(evaluate_imputation_file(reference, path.name))
            out_parts.append(evaluate_outlier_file(reference, path.name))
            norm_parts.append(evaluate_normalization_file(reference, path.name))

            logs.append({
                "file": path.name,
                "status": "ok",
                "rows": len(reference),
                "note": "",
            })
        except Exception as exc:
            logs.append({
                "file": path.name,
                "status": "error",
                "rows": np.nan,
                "note": repr(exc),
            })
            print(f"    ERROR: {exc!r}")

    run_log = pd.DataFrame(logs)
    imputation_raw = pd.concat(imp_parts, ignore_index=True) if imp_parts else pd.DataFrame()
    outlier_raw = pd.concat(out_parts, ignore_index=True) if out_parts else pd.DataFrame()
    normalization_raw = pd.concat(norm_parts, ignore_index=True) if norm_parts else pd.DataFrame()

    imputation_summary = (
        imputation_raw
        .groupby(["missing_level", "method", "feature"], as_index=False)[["rmse", "mae", "wilcoxon_p"]]
        .mean(numeric_only=True)
        .sort_values(["missing_level", "method", "feature"])
        .reset_index(drop=True)
    )

    outlier_summary = (
        outlier_raw
        .groupby(["outlier_level", "method", "feature"], as_index=False)[
            ["variance_reduction_pct", "ks_stat", "ks_p"]
        ]
        .mean(numeric_only=True)
        .sort_values(["outlier_level", "method", "feature"])
        .reset_index(drop=True)
    )

    normalization_summary = (
        normalization_raw
        .groupby(["mv_method", "outlier_method", "scaler", "feature"], as_index=False)[
            ["variance_reduction_pct", "skewness_reduction", "kurtosis_reduction", "ks_stat", "ks_p"]
        ]
        .mean(numeric_only=True)
        .sort_values(["mv_method", "outlier_method", "scaler", "feature"])
        .reset_index(drop=True)
    )

    return {
        "run_log": run_log,
        "imputation_raw": imputation_raw,
        "imputation_summary": imputation_summary,
        "outlier_raw": outlier_raw,
        "outlier_summary": outlier_summary,
        "normalization_raw": normalization_raw,
        "normalization_summary": normalization_summary,
    }


In [ ]:
# Full stage-level benchmark.
d2_results = run_stage_benchmark("D2", D2_DIR)
cheating_results = run_stage_benchmark("Cheating", CHEATING_DIR)

print("\nSuccessful files:")
print("D2:", int((d2_results["run_log"]["status"] == "ok").sum()), "/", len(d2_results["run_log"]))
print("Cheating:", int((cheating_results["run_log"]["status"] == "ok").sum()), "/", len(cheating_results["run_log"]))


## 11. Paper-level summaries

The paper reports dataset-level values averaged across corruption levels, signal features, and recordings. The following helper reproduces those compact stage-level summaries from the detailed tables above.


In [ ]:
def compact_stage_summary(results: dict[str, pd.DataFrame]) -> dict[str, pd.DataFrame]:
    imp = (
        results["imputation_summary"]
        .groupby("method", as_index=False)[["rmse", "mae"]]
        .mean(numeric_only=True)
    )
    out = (
        results["outlier_summary"]
        .groupby("method", as_index=False)[["ks_stat"]]
        .mean(numeric_only=True)
    )
    norm = (
        results["normalization_summary"]
        .groupby("scaler", as_index=False)[["ks_stat", "variance_reduction_pct", "skewness_reduction", "kurtosis_reduction"]]
        .mean(numeric_only=True)
    )
    return {"imputation": imp, "outlier": out, "normalization": norm}


d2_compact = compact_stage_summary(d2_results)
cheating_compact = compact_stage_summary(cheating_results)

print("CHEATING - Imputation")
display(cheating_compact["imputation"])
print("CHEATING - Outlier handling")
display(cheating_compact["outlier"])
print("CHEATING - Normalization")
display(cheating_compact["normalization"])

print("D2 - Imputation")
display(d2_compact["imputation"])
print("D2 - Outlier handling")
display(d2_compact["outlier"])
print("D2 - Normalization")
display(d2_compact["normalization"])


## 12. Published Table 2 consistency check

The values below are the stage-level values reported in Table 2 of the paper. They are used **only as a validation target**; they are not used in the computation itself.


In [ ]:
PAPER_TABLE2 = pd.DataFrame([
    ["Imputation", "Mean", "RMSE", 133.21, 132.97],
    ["Imputation", "Mean", "MAE", 107.31, 104.28],
    ["Imputation", "LOCF", "RMSE", 30.38, 57.99],
    ["Imputation", "LOCF", "MAE", 12.79, 16.53],
    ["Imputation", "KNN (k=5)", "RMSE", 139.88, 107.11],
    ["Imputation", "KNN (k=5)", "MAE", 108.46, 70.27],
    ["Outlier handling", "Z-score", "KS", 0.1031, 0.0832],
    ["Outlier handling", "MAD", "KS", 0.0868, 0.1053],
    ["Outlier handling", "Isolation Forest", "KS", 0.0876, 0.0876],
    ["Normalization", "Min-Max", "KS", 0.012, 0.014],
    ["Normalization", "Z-score", "KS", 0.023, 0.027],
    ["Normalization", "Robust Scaling", "KS", 0.016, 0.018],
], columns=["stage", "method", "metric", "Cheating_paper", "D2_paper"])

display(PAPER_TABLE2)


In [ ]:
def _lookup_compact(compact, stage, method, metric):
    if stage == "Imputation":
        method_map = {"Mean": "mean", "LOCF": "locf", "KNN (k=5)": "knn(k=5)"}
        key = method_map[method]
        row = compact["imputation"][compact["imputation"]["method"].eq(key)]
        return float(row.iloc[0][metric.lower()]) if len(row) else np.nan
    if stage == "Outlier handling":
        method_map = {"Z-score": "zscore", "MAD": "mad", "Isolation Forest": "iforest"}
        key = method_map[method]
        row = compact["outlier"][compact["outlier"]["method"].eq(key)]
        return float(row.iloc[0]["ks_stat"]) if len(row) else np.nan
    if stage == "Normalization":
        method_map = {"Min-Max": "minmax", "Z-score": "zscore", "Robust Scaling": "robust"}
        key = method_map[method]
        row = compact["normalization"][compact["normalization"]["scaler"].eq(key)]
        return float(row.iloc[0]["ks_stat"]) if len(row) else np.nan
    return np.nan


validation = PAPER_TABLE2.copy()
for dataset, compact in [("Cheating", cheating_compact), ("D2", d2_compact)]:
    validation[f"{dataset}_computed"] = [
        _lookup_compact(compact, row.stage, row.method, row.metric)
        for row in validation.itertuples(index=False)
    ]
    validation[f"{dataset}_delta"] = validation[f"{dataset}_computed"] - validation[f"{dataset}_paper"]

display(validation)


## 13. Export of the two benchmark workbooks

Each workbook uses the same seven-sheet structure:

- `run_log`
- `imputation_raw`
- `imputation_summary`
- `outlier_raw`
- `outlier_summary`
- `normalization_raw`
- `normalization_summary`

This keeps the dataset-level summaries traceable back to individual recordings and features.


In [ ]:
def export_workbook(results: dict[str, pd.DataFrame], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    sheet_order = [
        "run_log",
        "imputation_raw",
        "imputation_summary",
        "outlier_raw",
        "outlier_summary",
        "normalization_raw",
        "normalization_summary",
    ]

    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        for sheet_name in sheet_order:
            results[sheet_name].to_excel(writer, sheet_name=sheet_name, index=False)

        # Compact, deterministic formatting for repository artifacts.
        for worksheet in writer.book.worksheets:
            worksheet.freeze_panes = "A2"
            worksheet.auto_filter.ref = worksheet.dimensions
            for column_cells in worksheet.columns:
                letter = column_cells[0].column_letter
                max_len = max(len(str(cell.value)) if cell.value is not None else 0 for cell in column_cells[:200])
                worksheet.column_dimensions[letter].width = min(max(max_len + 2, 10), 30)


cheating_output = RESULTS_DIR / "CUMULUS_Output_Cheating.xlsx"
d2_output = RESULTS_DIR / "CUMULUS_Output_d2.xlsx"

export_workbook(cheating_results, cheating_output)
export_workbook(d2_results, d2_output)

print("Written:")
print(" -", cheating_output.resolve())
print(" -", d2_output.resolve())


## 14. Stage-level result interpretation

The stage-level benchmark supports the findings reported in the paper:

- **Imputation:** LOCF provides the lowest reconstruction error across the evaluated missingness levels in both datasets.
- **Outlier handling:** performance is context-dependent; the preferred method varies by dataset and contamination level.
- **Normalization:** Min-Max scaling provides the most stable distributional behavior in the reported stage-level comparison, while RobustScaler remains relevant for heavier-tailed distributions.

The important methodological point is that these are **stage-level observations only**. They must not be treated as a substitute for the complete-pipeline analysis. The latter is evaluated separately in the second notebook.
